# Laboratorio Semana 6: Preparación de Datos y Benchmarking (CSV vs Parquet)
**Materia:** Fundamentos de Big Data (IDSD413) - Escuela Politécnica Nacional
**Profesor:** David Esteban Galarza García

En este laboratorio demostraremos empíricamente el problema del cuello de botella de I/O (Lectura/Escritura) en Big Data. Generaremos un dataset sintético masivo de transacciones de retail y compararemos el rendimiento de un formato basado en filas (CSV) frente a un formato columnar nativo de Big Data (Apache Parquet).

In [ ]:
import pandas as pd
import numpy as np
import time
import os

# Reto 1: Se aumenta de 5M a 15M de filas para probar el consumo de RAM
N_ROWS = 15_000_000

print(f"Generando {N_ROWS} registros en memoria RAM...")
start_time = time.time()

# Generación de datos de prueba para transacciones
np.random.seed(42)
df = pd.DataFrame({
    'id_transaccion': np.arange(N_ROWS),
    'id_cliente': np.random.randint(1, 100000, N_ROWS),
    'monto': np.random.uniform(5.0, 2500.0, N_ROWS),
    'categoria': np.random.choice(['Electronica', 'Ropa', 'Hogar', 'Alimentos', 'Juguetes'], N_ROWS),
    'estado': np.random.choice(['Completado', 'Pendiente', 'Cancelado', 'Reembolsado'], N_ROWS)
})

print(f"Dataset generado en {time.time() - start_time:.2f} segundos.")
# Revisamos cuánto pesa el DataFrame completo en RAM antes de pasarlo a disco
df.info(memory_usage='deep')

Generando 5000000 registros en memoria RAM...
Dataset generado en 4.02 segundos.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000000 entries, 0 to 4999999
Data columns (total 5 columns):
 #   Column          Dtype  
---  ------          -----  
 0   id_transaccion  int64  
 1   id_cliente      int64  
 2   monto           float64
 3   categoria       object 
 4   estado          object 
dtypes: float64(1), int64(2), object(2)
memory usage: 190.7+ MB


## 1. Prueba de Escritura y Compresión en Disco
Guardaremos exactamente el mismo DataFrame en ambos formatos para analizar los tiempos de escritura y la eficiencia del algoritmo de compresión (Snappy) que utiliza Parquet por defecto.

In [2]:
# 1.1 Escritura en CSV (Basado en filas, texto plano)
print("Escribiendo archivo CSV...")
start_csv_write = time.time()
df.to_csv('transacciones_epn.csv', index=False)
csv_write_time = time.time() - start_csv_write
print(f"Tiempo de escritura CSV: {csv_write_time:.2f} segundos")

# 1.2 Escritura en Parquet (Columnar, tipado fuerte, comprimido)
print("\nEscribiendo archivo Parquet...")
start_parquet_write = time.time()
df.to_parquet('transacciones_epn.parquet', engine='pyarrow')
parquet_write_time = time.time() - start_parquet_write
print(f"Tiempo de escritura Parquet: {parquet_write_time:.2f} segundos")

# 1.3 Comparación de peso en el almacenamiento
csv_size = os.path.getsize('transacciones_epn.csv') / (1024 * 1024)
parquet_size = os.path.getsize('transacciones_epn.parquet') / (1024 * 1024)

print("-" * 40)
print(f"Peso del CSV: {csv_size:.2f} MB")
print(f"Peso del Parquet: {parquet_size:.2f} MB")
print(f"Reducción de almacenamiento: {((csv_size - parquet_size) / csv_size) * 100:.2f}%")

Escribiendo archivo CSV...
Tiempo de escritura CSV: 28.72 segundos

Escribiendo archivo Parquet...
Tiempo de escritura Parquet: 2.69 segundos
----------------------------------------
Peso del CSV: 244.55 MB
Peso del Parquet: 75.14 MB
Reducción de almacenamiento: 69.28%


## 2. Prueba de Lectura Completa (Full Scan)
Evaluaremos cuánto tarda el procesador en levantar la totalidad del archivo desde el disco duro hacia la memoria RAM de Google Colab.

In [3]:
# Limpiamos la memoria para evitar sesgos en la prueba
del df
import gc
gc.collect()

# 2.1 Lectura de CSV
print("Leyendo CSV a memoria...")
start_csv_read = time.time()
df_csv = pd.read_csv('transacciones_epn.csv')
csv_read_time = time.time() - start_csv_read
print(f"Tiempo de lectura CSV: {csv_read_time:.2f} segundos")

# 2.2 Lectura de Parquet
print("\nLeyendo Parquet a memoria...")
start_parquet_read = time.time()
df_parquet = pd.read_parquet('transacciones_epn.parquet', engine='pyarrow')
parquet_read_time = time.time() - start_parquet_read
print(f"Tiempo de lectura Parquet: {parquet_read_time:.2f} segundos")

Leyendo CSV a memoria...
Tiempo de lectura CSV: 4.96 segundos

Leyendo Parquet a memoria...
Tiempo de lectura Parquet: 0.95 segundos


## 3. Ventaja Columnar: Consultas Selectivas (Column Pruning)
En Ciencia de Datos, rara vez analizamos todas las columnas a la vez. En este escenario, el equipo de analítica solo necesita calcular el total de ventas (columna `monto`).

Veremos cómo Parquet escanea *únicamente* los bytes de esa columna, mientras que CSV está obligado a leer el archivo entero.

In [4]:
# 3.1 Agregación selectiva en CSV
print("Calculando ventas totales desde CSV...")
start_csv_query = time.time()
# Pandas tiene que analizar todo el archivo para extraer una sola columna
suma_csv = pd.read_csv('transacciones_epn.csv', usecols=['monto'])['monto'].sum()
csv_query_time = time.time() - start_csv_query
print(f"Ventas (CSV): ${suma_csv:,.2f} | Tiempo: {csv_query_time:.4f} segundos")

# 3.2 Agregación selectiva en Parquet
print("\nCalculando ventas totales desde Parquet...")
start_parquet_query = time.time()
# PyArrow inyecta la proyección directamente a nivel de disco
suma_parquet = pd.read_parquet('transacciones_epn.parquet', columns=['monto'], engine='pyarrow')['monto'].sum()
parquet_query_time = time.time() - start_parquet_query
print(f"Ventas (Parquet): ${suma_parquet:,.2f} | Tiempo: {parquet_query_time:.4f} segundos")

print("-" * 40)
print(f"Aceleración de consulta selectiva: {csv_query_time / parquet_query_time:.2f} veces más rápido")

Calculando ventas totales desde CSV...
Ventas (CSV): $6,261,756,916.24 | Tiempo: 3.7876 segundos

Calculando ventas totales desde Parquet...
Ventas (Parquet): $6,261,756,916.24 | Tiempo: 0.1247 segundos
----------------------------------------
Aceleración de consulta selectiva: 30.37 veces más rápido


## 4. Reto en Vivo 1: Filtrado desde el Disco (Predicate Pushdown)
¿Qué pasa si nuestro modelo solo necesita los datos de 'Electronica'?
En CSV, la memoria RAM puede colapsar porque Pandas debe cargar todo el archivo antes de filtrar. En Parquet, el motor filtra los datos directamente en el disco duro y solo envía a la RAM lo que coincide.

In [5]:
# 1. El método tradicional (CSV)
print("Filtrando CSV (Carga total -> Filtro)...")
start_csv = time.time()
df_temp_csv = pd.read_csv('transacciones_epn.csv')
df_filtrado_csv = df_temp_csv[df_temp_csv['categoria'] == 'Electronica']
tiempo_csv = time.time() - start_csv
print(f"Tiempo CSV: {tiempo_csv:.4f} segundos | Filas: {len(df_filtrado_csv)}")

# Liberamos memoria
del df_temp_csv, df_filtrado_csv
import gc; gc.collect()

# 2. El método Big Data (Parquet con Predicate Pushdown)
print("\nFiltrando Parquet (Filtro en disco -> Carga parcial)...")
start_parquet = time.time()
df_filtrado_parquet = pd.read_parquet(
    'transacciones_epn.parquet',
    engine='pyarrow',
    filters=[('categoria', '=', 'Electronica')]
)
tiempo_parquet = time.time() - start_parquet
print(f"Tiempo Parquet: {tiempo_parquet:.4f} segundos | Filas: {len(df_filtrado_parquet)}")

print("-" * 40)
print(f"Aceleración: {tiempo_csv / tiempo_parquet:.2f}x más rápido")

Filtrando CSV (Carga total -> Filtro)...
Tiempo CSV: 3.9260 segundos | Filas: 1000773

Filtrando Parquet (Filtro en disco -> Carga parcial)...
Tiempo Parquet: 0.7426 segundos | Filas: 1000773
----------------------------------------
Aceleración: 5.29x más rápido


## 5. Reto en Vivo 2: Agrupación y Estadísticas (Group By)
El análisis exploratorio requiere agrupar datos. Veamos cómo impacta la lectura columnar al calcular el promedio de ventas agrupado por el estado de la transacción.

In [6]:
# 1. Agrupación en CSV
print("Agrupando desde CSV...")
start_csv_grp = time.time()
grupo_csv = pd.read_csv('transacciones_epn.csv').groupby('estado')['monto'].mean()
tiempo_csv_grp = time.time() - start_csv_grp
print(f"Tiempo CSV: {tiempo_csv_grp:.4f} segundos")

# 2. Agrupación en Parquet
print("\nAgrupando desde Parquet...")
start_parquet_grp = time.time()
grupo_parquet = pd.read_parquet(
    'transacciones_epn.parquet',
    columns=['estado', 'monto'],
    engine='pyarrow'
).groupby('estado')['monto'].mean()
tiempo_parquet_grp = time.time() - start_parquet_grp
print(f"Tiempo Parquet: {tiempo_parquet_grp:.4f} segundos")

print("-" * 40)
print("Resultados de la agregación:")
print(grupo_parquet)

Agrupando desde CSV...
Tiempo CSV: 4.0408 segundos

Agrupando desde Parquet...
Tiempo Parquet: 0.9192 segundos
----------------------------------------
Resultados de la agregación:
estado
Cancelado      1252.852489
Completado     1252.037502
Pendiente      1252.560976
Reembolsado    1251.954358
Name: monto, dtype: float64
